# Lesson 2.4 — Context Windows, Token Budgets, and Prompt Reuse

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Estimate token usage and plan a context budget
- Order prompts so stable prefixes are cacheable (prompt caching)
- Apply prompt reuse strategies: caching, modular prompts, and Skills

**You will produce:** A `token_budget` estimator and a cache-optimized prompt layout.

## Concepts

The **context window** is the maximum tokens a model can consider at once (input + output). Architects manage a **token budget**: static system/policy content + retrieved context + conversation + output must fit, and every token has cost and latency.

**Prompt caching** lets Claude reuse a previously processed, unchanged **prefix** across requests — so if you put stable content (system prompt, policy docs, few-shot examples) *first* and the varying user input *last*, repeated calls are cheaper and faster (lower time-to-first-token). This is exactly the fix in the exam's Sample 2.

Other reuse strategies: **modular prompts** (compose from shared, versioned blocks) and **Skills** (packaged, reusable instructions/capabilities Claude can load). Together they cut duplication, cost, and drift. Verify current caching mechanics and limits in the docs.

> **Reference:** [Anthropic — Prompt caching](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

# Confirm exact model IDs at https://docs.anthropic.com/en/docs/about-claude/models
MODELS = {
    "opus": "claude-opus-4-20250514",
    "sonnet": "claude-sonnet-4-20250514",
    "haiku": "claude-3-5-haiku-20241022",
}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role": "user", "content": prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Estimate a token budget

We use a rough chars/4 heuristic to estimate tokens (real tokenization differs — note the doc link). The point is budgeting discipline, not exactness.

In [ ]:
def est_tokens(text):
    return max(1, len(text) // 4)  # rough heuristic; verify with real tokenizer

parts = {
    "system_prompt": "You are a triage assistant." * 20,
    "policy_doc": "COVERAGE POLICY ..." * 400,
    "few_shot": "example..." * 60,
    "user_msg": "Member requests MRI after 8 weeks PT.",
}
budget = {k: est_tokens(v) for k, v in parts.items()}
total = sum(budget.values())
for k, v in budget.items(): print(f"{k:14} ~{v} tokens")
print(f"total input ~{total} tokens (+ output)")

## Code-Along: Step 2 — Lay out the prompt for maximum cache reuse

Stable content first, volatile content last. The static prefix (system + policy + examples) is identical across requests and therefore cacheable; only the user message changes.

In [ ]:
static_prefix = parts["system_prompt"] + parts["policy_doc"] + parts["few_shot"]
def build_request(user_msg):
    # Static prefix is cache-eligible; user_msg varies per call.
    return {"cacheable_prefix_tokens": est_tokens(static_prefix),
            "variable_tokens": est_tokens(user_msg)}

r1 = build_request("Case A: MRI after 8 weeks PT")
r2 = build_request("Case B: MRI with no PT history")
print(r1); print(r2)
print("Across 1000 calls, the cacheable prefix is processed once, not 1000x (big cost/latency win).")
# See https://docs.anthropic.com for cache_control usage on content blocks.

## Exercise

Model the savings: assume the cacheable prefix is ~the size you computed and that cached input tokens are much cheaper than fresh ones. Compute the per-1000-request cost **with and without** caching using placeholder prices, and state the break-even call count. Add a comment linking to the pricing/caching docs.

In [ ]:
# TODO: placeholder prices (verify at https://docs.anthropic.com)
FRESH_PER_1K = 0.003
CACHED_PER_1K = 0.0003  # TODO verify cache read price ratio
# TODO: compute cost with vs without caching over 1000 requests and the break-even point

## Key Takeaways

- Plan a token budget: static + retrieved + conversation + output must fit and all cost money.
- Put stable content first so prompt caching can reuse the prefix (Sample 2's answer).
- Modular prompts and Skills cut duplication, cost, and version drift.
- Estimate tokens for budgeting, but verify with a real tokenizer and current pricing.

---

**Phase complete!** Return to the [study plan README](../README.md) to continue.